In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-isl_o23e/unsloth_6d27d37470b6493e8ac2dd04349f3745
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-isl_o23e/unsloth_6d27d37470b6493e8ac2dd04349f3745
  Resolved https://github.com/unslothai/unsloth.git to commit d4a311d8e71692961e5da1d26d98197fde94f41a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 33.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.4/284.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.8 MB

In [1]:
1

1

In [3]:
import unsloth
import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-11-26 13:42:16.667975: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764164537.033372      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764164537.114184      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/input/codingchatbot/final_model",  # ← Your path
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2025.11.4: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

Unsloth 2025.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
model = FastLanguageModel.for_inference(model)

In [6]:
from datasets import load_dataset
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train")

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

In [7]:
# Apply Llama 3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


In [8]:
# ✅ Verify tokens are different
print(f"Pad Token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS Token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

Pad Token: <|PAD_TOKEN|> (ID: 151665)
EOS Token: <|im_end|> (ID: 151643)


In [29]:
query = dataset[10900]
query

{'output': 'const array = [1, 2, 3, 4];\nconst sum = array[1] + array[2];\nconsole.log(sum); // 5',
 'instruction': 'Modify the following array to output the sum of the second and third items in the array.',
 'input': 'const array = [1, 2, 3, 4];'}

In [ ]:
# def format_codealpaca(examples):
#     conversations = []
#     for instruction, input_text, output in zip(
#         examples["instruction"],
#         examples["input"],
#         examples["output"]
#     ):
#         if input_text and input_text.strip():
#             user_message = f"{instruction}\n\nInput:\n{input_text}"
#         else:
#             user_message = instruction
#         conversation = [
#             {"role": "user", "content": user_message},
#             {"role": "assistant", "content": output}
#         ]
#         conversations.append(conversation)
    
#     # Apply chat template and REMOVE trailing whitespace
#     texts = [
#         tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False).rstrip()
#         for conv in conversations
#     ]
   
#     return {"text": texts}

In [30]:
user_message = f"{query['instruction']}\n\nInput:\n{query['input']}"


In [31]:
messages = [
    {"role": "user", "content": user_message}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.5,
    top_p=0.9,
    do_sample=True,
    # ✅ CRITICAL: Proper EOS handling
    eos_token_id=tokenizer.eos_token_id,  # Stop at <|im_end|>
    pad_token_id=tokenizer.pad_token_id,  # Different from EOS now!
    # Optional but helpful:
    repetition_penalty=1.1,  # Slight penalty for repetition
)
# The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.

In [32]:
# Decode only the generated part
generated_ids = outputs[0][inputs.shape[1]:]
result = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(result)

const array = [1, 2, 3, 4];

let sum = array[1] + array[2];
console.log(sum); // Output: 5


In [28]:
# Decode and print
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

user
Write a JavaScript function to convert a number from octal to decimal.

Input:
octal_number = 014
assistant
function octalToDecimal(octal) {
    let result = parseInt(octal, 8);
    return result;
} 

console.log(octalToDecimal(014)); // Output: 12
